In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
master = pd.read_csv("../Data/Processed/final_master_dataset_normalized.csv")

In [3]:
master.shape

(36551, 29)

In [4]:
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 36551 entries, 0 to 36550
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   36551 non-null  str    
 1   player_name            36551 non-null  str    
 2   acwr                   36551 non-null  float64
 3   atl                    36551 non-null  float64
 4   ctl28                  36551 non-null  float64
 5   ctl42                  36551 non-null  float64
 6   daily_load             36551 non-null  float64
 7   monotony               36551 non-null  float64
 8   strain                 36551 non-null  float64
 9   weekly_load            36551 non-null  float64
 10  fatigue                16993 non-null  float64
 11  mood                   17000 non-null  float64
 12  readiness              16998 non-null  float64
 13  sleep_duration         16982 non-null  float64
 14  sleep_quality          16995 non-null  float64
 15  soreness     

In [5]:
model_data = master.dropna(subset=["fatigue"])

In [6]:
model_data.shape

(16993, 29)

In [7]:
y = model_data["fatigue"]

In [8]:
X = model_data.drop(
    columns=[
        "fatigue",
        "team_performance",
        "offensive_performance",
        "defensive_performance",
        "player_name",
        "date"
    ]
)

In [9]:
missing = (
    X.isnull()
     .mean()
     .mul(100)
     .sort_values(ascending=False)
)

missing

avg_heart_rate    91.031601
max_heart_rate    91.031601
avg_accel_x       91.031601
avg_accel_y       91.031601
avg_accel_z       91.031601
rows              91.031601
avg_speed         91.031601
injured           91.031601
max_speed         91.031601
sleep_duration     0.147119
sleep_quality      0.064733
stress             0.052963
readiness          0.052963
mood               0.041193
soreness           0.035309
strain             0.000000
monotony           0.000000
daily_load         0.000000
ctl42              0.000000
ctl28              0.000000
atl                0.000000
acwr               0.000000
weekly_load        0.000000
dtype: float64

In [10]:
gps_columns = [
    "avg_speed",
    "max_speed",
    "avg_heart_rate",
    "max_heart_rate",
    "avg_accel_x",
    "avg_accel_y",
    "avg_accel_z",
    "rows",
    "injured"
]

X = X.drop(columns=gps_columns)

In [11]:
X.isnull().sum()

acwr               0
atl                0
ctl28              0
ctl42              0
daily_load         0
monotony           0
strain             0
weekly_load        0
mood               7
readiness          9
sleep_duration    25
sleep_quality     11
soreness           6
stress             9
dtype: int64

In [13]:
from sklearn.model_selection import train_test_split
import logging

logger = logging.getLogger(__name__)
logger.info("Splitting dataset into train and test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (13594, 14)
X_test  : (3399, 14)
y_train : (13594,)
y_test  : (3399,)


In [14]:
from sklearn.impute import SimpleImputer

logger.info("Performing Median Imputation...")

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X.columns
)

In [15]:
print("Train Missing :", X_train.isnull().sum().sum())
print("Test Missing  :", X_test.isnull().sum().sum())

Train Missing : 0
Test Missing  : 0


In [16]:
import joblib

joblib.dump(
    imputer,
    "../model/imputer.pkl"
)

logger.info("Imputer Saved.")

In [17]:
X_train.to_csv("../Data/Processed/X_train.csv", index=False)
X_test.to_csv("../Data/Processed/X_test.csv", index=False)

y_train.to_csv("../Data/Processed/y_train.csv", index=False)
y_test.to_csv("../Data/Processed/y_test.csv", index=False)

logger.info("Train/Test datasets saved successfully.")